In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Take a quick look at the data

In [2]:
# Data: https://www.kaggle.com/competitions/titanic/overview

In [ ]:
#Hello
# data = pd.read_csv('data/titanic.csv')

In [4]:
print("First few rows of the dataset and shape:")
print(data.shape)
data.head()

First few rows of the dataset and shape:
(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
print("\nData structure information:")
print(data.info())

In [ ]:
print("\nSummary statistics of numerical columns:")
print(data.describe())

In [ ]:
print("\nSummary statistics of categorical columns:")
print(data.describe(include='O'))

In [ ]:
print("\nData types of each column:")
print(data.dtypes)

In [ ]:
print("\nUnique values in each column:")
print(data.nunique())

In [ ]:
print("\nValue counts for the 'Sex' column:")
print(data['Sex'].value_counts())

In [ ]:
print("\nNumber of duplicate rows:")
print(data.duplicated().sum())

# Split the data into Training, Validation, and Test sets

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier

In [ ]:
X = data.drop('Survived', axis=1)
y = data['Survived']

### Random Split

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("\nValue Counts for Target Variable (Random Split):")
print("Training set (Survived):")
print(y_train.value_counts(normalize=True)) 
print("\nValidation set (Survived):")
print(y_val.value_counts(normalize=True))    
print("\nTest set (Survived):")
print(y_test.value_counts(normalize=True))   

### Stratified Split

In [ ]:
X_train_strat, X_temp_strat, y_train_strat, y_temp_strat = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val_strat, X_test_strat, y_val_strat, y_test_strat = train_test_split(X_temp_strat, y_temp_strat, test_size=0.5, random_state=42, stratify=y_temp_strat)

print("\nValue Counts for Target Variable (Stratified Split):")
print("Training set (Survived):")
print(y_train_strat.value_counts(normalize=True))
print("\nValidation set (Survived):")
print(y_val_strat.value_counts(normalize=True))  
print("\nTest set (Survived):")
print(y_test_strat.value_counts(normalize=True)) 

### Cross-validation (Data Leakage)

##### Wrong

In [ ]:
X_clean = X.copy()
y_clean = y.copy()

dummy_clf = DummyClassifier(strategy='most_frequent')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cross_val_scores = cross_val_score(dummy_clf, X_clean, y_clean, cv=skf)

print("\nCross-Validation Scores:")
print(cross_val_scores)

##### Right

In [ ]:
dummy_clf = DummyClassifier(strategy='most_frequent')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

class DataCleaner:
    def fit(self, X, y=None):
        pass
    
    def transform(self, X):
        return X

cross_val_scores = []

for train_index, val_index in skf.split(X, y):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    cleaner = DataCleaner()

    cleaner.fit(X_train)

    X_train_cleaned = cleaner.transform(X_train)
    X_val_cleaned = cleaner.transform(X_val)

    dummy_clf.fit(X_train_cleaned, y_train)

    score = dummy_clf.score(X_val_cleaned, y_val)
    cross_val_scores.append(score)

print("\nCross-Validation Scores:")
print(cross_val_scores)

___

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [ ]:
print(f"Training set (X_train, y_train) shape: {X_train.shape}, {y_train.shape}")
print(f"Validation set (X_val, y_val) shape: {X_val.shape}, {y_val.shape}")
print(f"Test set (X_test, y_test) shape: {X_test.shape}, {y_test.shape}")

In [ ]:
X_train_copy = X_train.copy()
y_train_copy = y_train.copy()

___

# Exploratory Data Analysis (EDA)

### Analyze the target variable

In [ ]:
print("Target variable distribution (Survived):")
print(y_train.value_counts(normalize=True))
sns.countplot(x=y_train, hue=y_train)
plt.title('Count of Survivors')
plt.show()

### Assess Missing Values

In [ ]:
print("\nMissing Values in Each Column:")
print(X_train.isnull().sum())

In [ ]:
sns.heatmap(X_train.isnull(), cbar=False, cmap='binary')
plt.title('Missing Values Heatmap')
plt.show()

### Explore relationships between features and the target variable

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x='Sex', y=y_train, data=X_train, hue='Sex')
plt.title('Survival Rate by Gender')
plt.ylabel('Survival Rate')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x='Pclass', y=y_train, data=X_train, hue='Pclass')
plt.title('Survival Rate by Passenger Class')
plt.ylabel('Survival Rate')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=X_train, x='Age', hue=y_train, bins=30, kde=True, 
             palette='pastel', stat='density', common_norm=False)

plt.title('Age Distribution by Survival Status')
plt.xlabel('Age')
plt.ylabel('Density')
plt.legend(title='Survived', labels=['Not Survived (0)', 'Survived (1)'])
plt.show()

In [ ]:
# Violinplot Displays distribution of data  
# across all levels of a category. 
sns.violinplot(x ="Sex", y ="Age", hue ="Survived",  
data = pd.concat([X_train, y_train], axis=1), split = True) 

Exercises:


Given the Titanic dataset, perform the following tasks:

1.Categorize Fare into Quantiles:

- Use the qcut function from the Pandas library to divide the 'Fare' column into four equal-sized bins.
- Assign appropriate labels to these bins.

2.Visualize Survival Rates by Fare Range:

- Utilize Seaborn's barplot function to display the survival rates across the different fare ranges.
- Ensure that the x-axis represents the fare ranges, and the y-axis represents the survival rate.

Note: use  **barplot** 

<details>

<summary><strong>Solution</strong> (show)</summary>

```python
# Divide Fare into 4 bins 
X_train['Fare_Range'] = pd.qcut(X_train['Fare'], 4) 
  
# Barplot - Shows approximate values based  
# on the height of bars. 
sns.barplot(x ='Fare_Range', y ='Survived',  
data = pd.concat([X_train, y_train], axis=1)) 
</details>

```

### Explore relationships between features and other features

corr

In [ ]:
numerical_features = X_train.select_dtypes(include=['int64', 'float64'])

corr_matrix = numerical_features.corr()

print("Correlation Matrix:")
print(corr_matrix)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Heatmap of Numerical Features")
plt.show()

### Split variables into numerical and categorical

In [ ]:
numerical_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

print("\nNumerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

### Numerical Features

Discrete Features

In [ ]:
discrete_var = [var for var in numerical_features if X_train[var].nunique() < 20]
print(len(discrete_var))
X_train[discrete_var].head()

In [ ]:
for var in discrete_var:
    ct = pd.crosstab(X_train[var], y_train)

    ct.plot(kind='bar')
    plt.xlabel(var)
    plt.ylabel('Count')
    plt.title(f'Counts of {var} Variable Grouped by Target')
    plt.show()

Exercises:


Using the Titanic dataset, create a count plot that shows the number of passengers who embarked from each port ('Embarked'), differentiated by their survival status ('Survived'), and further separated by passenger class ('Pclass').

Note: use  **catplot** 

<details>

<summary><strong>Solution</strong> (show)</summary>

```python
# Countplot 
sns.catplot(x ='Embarked', hue ='Survived',  
kind ='count', col ='Pclass', data = pd.concat([X_train, y_train], axis=1)) 
</details>

```

Continuous Feature

In [ ]:
continuous_var = [var for var in numerical_features if var not in discrete_var + ['PassengerId']]
print(len(continuous_var))
X_train[continuous_var].head()

In [ ]:
data_with_target = pd.concat([X_train[continuous_var], y_train], axis=1)

sns.pairplot(data_with_target, hue='Survived', height=5)

In [ ]:
X_train[continuous_var].hist(bins=30, figsize=(10, 5));

### Categorical Feature

In [ ]:
print(len(categorical_features))
X_train[categorical_features].head()

Number of labels: Cardinality

In [ ]:
cardinalith = X_train[categorical_features].nunique().sort_values(ascending=False)
print(cardinalith / X_train.shape[0])
cardinalith.plot.bar();

# Data Cleaning

### Type Conversion

In [ ]:
print("\nData Types Before Conversion:")
print(X_train.dtypes)

type_mapping = {
    'PassengerId': int,
    'Pclass': object,
    'Name': object,
    'Sex': object,
    'Age': float,
    'SibSp': int,
    'Parch': int,
    'Ticket': object,
    'Fare': float,
    'Cabin': object,
    'Embarked': object
}

for column, dtype in type_mapping.items():
    X_train[column] = X_train[column].astype(dtype)

print("\nData Types After Conversion:")
print(X_train.dtypes)

### Filter Bad Data

In [ ]:
X_train_filter = X_train.copy()

In [ ]:
original_shape = X_train_filter.shape
print("\nOriginal shape of the data:")
print(original_shape)

bad_age_condition = (X_train_filter['Age'] < 5) | (X_train_filter['Age'] > 70)
bad_fare_condition = X_train_filter['Fare'] <= 0

filtered_data = X_train_filter[~(bad_age_condition | bad_fare_condition)]

filtered_shape = filtered_data.shape
print("\nShape of the data after filtering bad data:")
print(filtered_shape)

filtered_rows = original_shape[0] - filtered_shape[0]
print("\nNumber of rows filtered out:")
print(filtered_rows)

### Replace and Map Categories to Fix Them

In [ ]:
X_train['Sex'] = X_train['Sex'].replace({'male': 'Male', 'female': 'Female', 'F': 'Female', 'M': 'Male'})

print("\nUnique values in the 'Sex' column after mapping:")
print(X_train['Sex'].unique())

### Drop Unnecessary Features

In [ ]:
y_train = pd.DataFrame(y_train)
y_train.index = X_train['PassengerId']

In [ ]:
X_train.set_index('PassengerId', inplace=True)  # Set 'PassengerId' as index
X_train.drop(columns=['Cabin', 'Ticket'], inplace=True)  # Drop unnecessary columns

print("\nShape of the data after dropping unnecessary features:")
print(X_train.shape)

In [ ]:
print("X:")
X_train.head()

In [ ]:
print("y:")
y_train.head()

___

# Feature Engineering and Transformation

### Transformations Applied to the Target Variable (y)

In [ ]:
from sklearn.preprocessing import PowerTransformer
y_fake = X_train['Fare']

plt.figure(figsize=(14, 6))

plt.subplot(1, 3, 1)
plt.hist(y_fake, bins=30, color='blue', alpha=0.7)
plt.title('Original Fare Distribution')
plt.xlabel('Fare')
plt.ylabel('Frequency')

y_log = np.log1p(y_fake)

plt.subplot(1, 3, 2)
plt.hist(y_log, bins=30, color='green', alpha=0.7)
plt.title('Log Transformed Fare Distribution')
plt.xlabel('Log(Fare + 1)')
plt.ylabel('Frequency')

# Apply PowerTransformer Transformation
pt = PowerTransformer()
y_yeojohnson = pt.fit_transform(y_fake.values.reshape(-1, 1)).flatten()

# Fifth subplot: Yeo-Johnson Transformed Fare Distribution
plt.subplot(1, 3, 3)
plt.hist(y_yeojohnson, bins=30, color='orange', alpha=0.7)
plt.title('Yeo-Johnson Transformed Fare Distribution')
plt.xlabel('Yeo-Johnson Transformed Fare')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

Transformations vs Scale

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import boxcox
from sklearn.preprocessing import PowerTransformer, Normalizer

# Example: Load a sample dataset (you can replace this with your actual dataset)

# Extract the 'Fare' column
y_fake = X_train['Fare']

# Create a figure with a specified size
plt.figure(figsize=(14, 6))

# First subplot: Original Fare Distribution
plt.subplot(1, 2, 1)
plt.hist(y_fake, bins=30, color='blue', alpha=0.7)
plt.title('Original Fare Distribution')
plt.xlabel('Fare')
plt.ylabel('Frequency')

# Apply Normalizer
pt = Normalizer()
y_yeojohnson = pt.fit_transform(y_fake.values.reshape(-1, 1)).flatten()

# Fifth subplot:
plt.subplot(1, 2, 2)
plt.hist(y_yeojohnson, bins=30, color='orange', alpha=0.7)
plt.title('Normalizer Distribution')
plt.xlabel('Normalizer')
plt.ylabel('Frequency')

# Adjust the layout to prevent overlap
plt.tight_layout()

# Display the plots
plt.show()

Exercises:

Try QuantileTransformer

<details>

<summary><strong>Solution</strong> (show)</summary>

```python
from scipy.stats import boxcox
from sklearn.preprocessing import QuantileTransformer
# Extract the 'Fare' column
y_fake = X_train['Fare']

# Create a figure with a specified size
plt.figure(figsize=(24, 6))

# First subplot: Original Fare Distribution
plt.subplot(1, 2, 1)
plt.hist(y_fake, bins=30, color='blue', alpha=0.7)
plt.title('Original Fare Distribution')
plt.xlabel('Fare')
plt.ylabel('Frequency')

# Apply Quantile Transformation
qt = QuantileTransformer(n_quantiles=10, random_state=0, output_distribution='normal')
y_quantile = qt.fit_transform(y_fake.values.reshape(-1, 1)).flatten()

# Plot the transformed data
plt.subplot(1, 2,2)
plt.hist(y_quantile, bins=30, color='cyan', alpha=0.7)
plt.title('Quantile Transformed Fare Distribution')
plt.xlabel('Quantile Transformed Fare')
plt.ylabel('Frequency')



# Adjust the layout to prevent overlap
plt.tight_layout()

# Display the plots
plt.show()
</details>

```

### Imputing Missing Values

In [ ]:
from sklearn.impute import SimpleImputer

print("\nMissing Values in Each Column Before Imputation:")
print(X_train.isnull().sum())

age_imputer = SimpleImputer(strategy='median')
embarked_imputer = SimpleImputer(strategy='most_frequent')

X_train['Age'] = age_imputer.fit_transform(X_train[['Age']])

X_train['Embarked'] = embarked_imputer.fit_transform(X_train[['Embarked']]).flatten()

print("\nMissing Values in Each Column After Imputation:")
print(X_train.isnull().sum())

### Creating New Feature

In [ ]:
X_train['FamilySize'] = X_train['SibSp'] + X_train['Parch'] + 1

X_train['Title'] = X_train['Name'].str.extract(' ([A-Za-z]+)\.')

title_mapping = {
    'Mr': 'Mr',
    'Mrs': 'Mrs',
    'Miss': 'Miss',
    'Master': 'Master',
    'Dr': 'Rare',
    'Rev': 'Rare',
    'Col': 'Rare',
    'Mlle': 'Miss',
    'Ms': 'Mrs',
    'Mme': 'Mrs',
    'Don': 'Rare',
    'Dona': 'Rare',
    'Jonkheer': 'Rare',
    'Lady': 'Rare',
    'Sir': 'Rare'
}
X_train['Title'] = X_train['Title'].map(title_mapping)
X_train['Title'] = X_train['Title'].fillna('Rare')

X_train.head()

Exercises:

Create a new feature 'IsAlone' 

<details>

<summary><strong>Solution</strong> (show)</summary>

```python
X_train['IsAlone'] = 1
X_train['IsAlone'].loc[X_train['FamilySize'] > 1] = 0
</details>

```

Exercises:

Create a 'AgeGroup' feature by binning the 'Age' column

<details>

<summary><strong>Solution</strong> (show)</summary>

```python
bins = [0, 12, 18, 35, 60, np.inf]
labels = ['Child', 'Teenager', 'Young Adult', 'Adult', 'Senior']
X_train['AgeGroup'] = pd.cut(X_train['Age'], bins=bins, labels=labels)
</details>

```

Exercises:

Create a 'FareGroup' feature by binning the 'Fare' column

<details>

<summary><strong>Solution</strong> (show)</summary>

```python
data['FareGroup'] = pd.qcut(data['Fare'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])
</details>

```

### Drop features

In [ ]:
from feature_engine.selection import DropFeatures

drop_features = DropFeatures(features_to_drop=['Name', 'SibSp', 'Parch'])

X_train = drop_features.fit_transform(X_train)

X_train.head()

### Encoding categorical variables


In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

one_hot_encoder = OneHotEncoder(drop='first', sparse_output=False)

X_train_embarked_encoded = one_hot_encoder.fit_transform(X_train[['Embarked']])

X_train_embarked_encoded = pd.DataFrame(X_train_embarked_encoded, columns=one_hot_encoder.get_feature_names_out(['Embarked']), index=X_train.index)

X_train = pd.concat([X_train.drop('Embarked', axis=1), X_train_embarked_encoded], axis=1)

In [ ]:
label_encoder_sex = LabelEncoder()
label_encoder_title = LabelEncoder()

X_train['Sex'] = label_encoder_sex.fit_transform(X_train['Sex'])
X_train['Title'] = label_encoder_title.fit_transform(X_train['Title'])

# Feature scaling

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

standard_scaler = StandardScaler()

standard_scaler_features = ['Pclass', 'Age', 'Sex', 'FamilySize', 'Title', 'Embarked_Q', 'Embarked_S']

X_train[standard_scaler_features] = standard_scaler.fit_transform(X_train[standard_scaler_features])

In [ ]:
min_max_scaler = MinMaxScaler()
min_max_scaler_features = ['Fare']
X_train[min_max_scaler_features] = min_max_scaler.fit_transform(X_train[min_max_scaler_features])

In [ ]:
X_train

Exercises:

Try RobustScaler

<details>

<summary><strong>Solution</strong> (show)</summary>

```python
from sklearn.preprocessing import RobustScaler

robust_scaler = RobustScaler()
robust_scaler_features = ['Fare']
X_train[robust_scaler_features] = robust_scaler.fit_transform(X_train[robust_scaler_features])
</details>

```

# Final Data Preparation


### Build transformation pipelines for ml eng

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class TypeConverter(BaseEstimator, TransformerMixin):
    def __init__(self, type_mapping):
        self.type_mapping = type_mapping
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        for column, dtype in self.type_mapping.items():
            X_copy[column] = X_copy[column].astype(dtype)
        return X_copy

class SetIndexByPassengerId(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        
        X_copy.set_index('PassengerId', inplace=True)
        
        return X_copy
    
class SexMappingTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, sex_col):
        self.mapping = {'male': 'Male', 'female': 'Female', 'F': 'Female', 'M': 'Male'}
        self.sex_col = sex_col
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        
        X_copy[self.sex_col] = X_copy[self.sex_col].replace(self.mapping)
        return X_copy
    
class ColumnImputer(BaseEstimator, TransformerMixin):
    def __init__(self, strategy='mean', column_name=None):
        self.strategy = strategy
        self.column_name = column_name
        self.imputer = SimpleImputer(strategy=self.strategy)
        
    def fit(self, X, y=None):
        self.imputer.fit(X[[self.column_name]])
        return self
    ""
    def transform(self, X):
        X_copy = X.copy()
        if self.strategy == 'most_frequent':
            X_copy[self.column_name] = self.imputer.transform(X[[self.column_name]]).flatten()
        else:
            X_copy[self.column_name] = self.imputer.transform(X[[self.column_name]])
        return X_copy
    
class FeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self, add_title=True, add_family_size=True):
        self.add_title = add_title
        self.add_family_size = add_family_size
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        
        if self.add_family_size:
            X_copy['FamilySize'] = X_copy['SibSp'] + X_copy['Parch'] + 1
        
        if self.add_title:
            X_copy['Title'] = X_copy['Name'].str.extract(' ([A-Za-z]+)\.')
            
            title_mapping = {
                'Mr': 'Mr',
                'Mrs': 'Mrs',
                'Miss': 'Miss',
                'Master': 'Master',
                'Dr': 'Rare',
                'Rev': 'Rare',
                'Col': 'Rare',
                'Mlle': 'Miss',
                'Ms': 'Mrs',
                'Mme': 'Mrs',
                'Don': 'Rare',
                'Dona': 'Rare',
                'Jonkheer': 'Rare',
                'Lady': 'Rare',
                'Sir': 'Rare'
            }
            X_copy['Title'] = X_copy['Title'].map(title_mapping)
            X_copy['Title'] = X_copy['Title'].fillna('Rare')  # Fill missing titles as 'Rare'
        
        return X_copy
    
class OneHotEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column=None, drop_first=True):
        self.column = column
        self.drop_first = drop_first
        self.encoder = OneHotEncoder(drop='first', sparse_output=False)
        
    def fit(self, X, y=None):
        if self.column:
            self.encoder.fit(X[self.column])
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        if self.column:
            X_encoded = self.encoder.transform(X_copy[self.column])
            encoded_df = pd.DataFrame(X_encoded, columns=self.encoder.get_feature_names_out(self.column), index=X_copy.index)
            X_copy = pd.concat([X_copy.drop(self.column, axis=1), encoded_df], axis=1)
        return X_copy
    
class LabelEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column=None):
        self.column = column
        self.encoder = LabelEncoder()

    def fit(self, X, y=None):
        if self.column:
            self.encoder.fit(X[self.column])
        return self

    def transform(self, X):
        X_copy = X.copy()
        if self.column:
            X_copy[self.column] = self.encoder.transform(X_copy[self.column])
        return X_copy
    
class ScalerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, standard_scaler_features=None, min_max_scaler_features=None):
        self.standard_scaler_features = standard_scaler_features
        self.min_max_scaler_features = min_max_scaler_features
        self.standard_scaler = StandardScaler()
        self.min_max_scaler = MinMaxScaler()

    def fit(self, X, y=None):
        if self.standard_scaler_features:
            self.standard_scaler.fit(X[self.standard_scaler_features])
        
        if self.min_max_scaler_features:
            self.min_max_scaler.fit(X[self.min_max_scaler_features])

        return self

    def transform(self, X):
        X_copy = X.copy()
        
        if self.standard_scaler_features:
            X_copy[self.standard_scaler_features] = self.standard_scaler.transform(X_copy[self.standard_scaler_features])
        
        if self.min_max_scaler_features:
            X_copy[self.min_max_scaler_features] = self.min_max_scaler.transform(X_copy[self.min_max_scaler_features])

        return X_copy

In [ ]:
type_mapping = {
    'PassengerId': int,
    'Pclass': object,
    'Name': object,
    'Sex': object,
    'Age': float,
    'SibSp': int,
    'Parch': int,
    'Ticket': object,
    'Fare': float,
    'Cabin': object,
    'Embarked': object
}

sex_mapping = 'Sex'
unnecessary_features = ['Cabin', 'Ticket']
median_imputer = 'Age'
most_freq_imputer = 'Embarked'
add_title = True
add_family_size = True
features_to_drop = ['Name', 'SibSp', 'Parch']
embarked_encoder = 'Embarked'
sex_encoder = 'Sex'
title_encoder = 'Title'
standard_scaler_features = ['Pclass', 'Age', 'Sex', 'FamilySize', 'Title', 'Embarked_Q', 'Embarked_S']
min_max_scaler_features = ['Fare']

In [ ]:
from sklearn.pipeline import Pipeline
from feature_engine.selection import DropFeatures

y_train_cleaned = pd.DataFrame(y_train_copy)
y_train_cleaned.index = X_train_copy['PassengerId']

pipeline = Pipeline(steps=[
    ('type_converter', TypeConverter(type_mapping)),
    ('sex_mapping', SexMappingTransformer(sex_col=sex_mapping)),
    ('set_index', SetIndexByPassengerId()),
    ('drop_unnecessary_features', DropFeatures(features_to_drop=unnecessary_features)),
    ('age_imputer', ColumnImputer(strategy='median', column_name=median_imputer)),
    ('embarked_imputer', ColumnImputer(strategy='most_frequent', column_name=most_freq_imputer)),
    ('feature_engineering', FeatureEngineering(add_title=add_title, add_family_size=add_family_size)),
    ('drop_features', DropFeatures(features_to_drop=features_to_drop)),
    ('onehot_encoder', OneHotEncoderTransformer(column=[embarked_encoder])),
    ('sex_label_encoder', LabelEncoderTransformer(column=sex_encoder)),
    ('title_label_encoder', LabelEncoderTransformer(column=title_encoder)),
    ('scaler', ScalerTransformer(standard_scaler_features=standard_scaler_features,
                                 min_max_scaler_features=min_max_scaler_features)),
])

X_train_cleaned = pipeline.fit_transform(X_train_copy)

In [ ]:
assert X_train_cleaned.equals(X_train), "X_train and X_train_copy are not the same after transformation!"
assert y_train_cleaned.equals(y_train), "y_train and y_train_copy are not the same after transformation!"

---

# Model Selection and Training

### Select appropriate algorithms using validation

In [ ]:
y_val_cleaned = pd.DataFrame(y_val)
y_val_cleaned.index = X_val['PassengerId']
X_val_cleaned = pipeline.transform(X_val)

1. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

log_reg_model = LogisticRegression()
log_reg_model.fit(X_train_cleaned, y_train_cleaned.values.ravel())

y_train_pred_log_reg = log_reg_model.predict(X_train_cleaned)
y_val_pred_log_reg = log_reg_model.predict(X_val_cleaned)

acc_train_log_reg = accuracy_score(y_train_cleaned, y_train_pred_log_reg)
acc_val_log_reg = accuracy_score(y_val_cleaned, y_val_pred_log_reg)

print("Logistic Regression Accuracy:")
print(f"Training Accuracy: {acc_train_log_reg:.4f}")
print(f"Validation Accuracy: {acc_val_log_reg:.4f}")


2. K-Nearest Neighbors (KNN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier()
knn_model.fit(X_train_cleaned, y_train_cleaned.values.ravel())

y_train_pred_knn = knn_model.predict(X_train_cleaned)
y_val_pred_knn = knn_model.predict(X_val_cleaned)

acc_train_knn = accuracy_score(y_train_cleaned, y_train_pred_knn)
acc_val_knn = accuracy_score(y_val_cleaned, y_val_pred_knn)

print("KNN Accuracy:")
print(f"Training Accuracy: {acc_train_knn:.4f}")
print(f"Validation Accuracy: {acc_val_knn:.4f}")

3. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X_train_cleaned, y_train_cleaned.values.ravel())

y_train_pred_dt = decision_tree_model.predict(X_train_cleaned)
y_val_pred_dt = decision_tree_model.predict(X_val_cleaned)

acc_train_dt = accuracy_score(y_train_cleaned, y_train_pred_dt)
acc_val_dt = accuracy_score(y_val_cleaned, y_val_pred_dt)

print("Decision Tree Accuracy:")
print(f"Training Accuracy: {acc_train_dt:.4f}")
print(f"Validation Accuracy: {acc_val_dt:.4f}")

4. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(random_state=42)
random_forest_model.fit(X_train_cleaned, y_train_cleaned.values.ravel())

y_train_pred_rf = random_forest_model.predict(X_train_cleaned)
y_val_pred_rf = random_forest_model.predict(X_val_cleaned)

acc_train_rf = accuracy_score(y_train_cleaned, y_train_pred_rf)
acc_val_rf = accuracy_score(y_val_cleaned, y_val_pred_rf)

print("Random Forest Accuracy:")
print(f"Training Accuracy: {acc_train_rf:.4f}")
print(f"Validation Accuracy: {acc_val_rf:.4f}")

### Analyze the best models and their errors

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

y_val_pred_knn = knn_model.predict(X_val_cleaned)

conf_matrix = confusion_matrix(y_val_cleaned, y_val_pred_knn)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Not Survived', 'Survived'], yticklabels=['Not Survived', 'Survived'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix for KNN')
plt.show()

print(classification_report(y_val_cleaned, y_val_pred_knn))

### Train models using pipeline

In [ ]:
data = pd.read_csv('../data/titanic.csv')

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [ ]:
combined_pipeline = Pipeline(steps=[
    ('preprocessing', pipeline),
    ('model', KNeighborsClassifier())
])

In [ ]:
combined_pipeline.fit(X_train, y_train)

In [ ]:
y_train_pred = combined_pipeline.predict(X_train)
y_val_pred = combined_pipeline.predict(X_val)

train_acc = accuracy_score(y_train, y_train_pred)
val_acc = accuracy_score(y_val, y_val_pred)

print("\nKNN Model Accuracy:")
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

# Fine-Tuning the Model

### Grid Search

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_neighbors': [3, 5, 7, 9],
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(
    estimator=combined_pipeline,
    param_grid=param_grid,
    cv=5,  
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters (Grid Search):", grid_search.best_params_)
print("Best Accuracy (Grid Search):", grid_search.best_score_)

best_pipeline_grid = grid_search.best_estimator_
y_val_pred_grid = best_pipeline_grid.predict(X_val)
val_acc_grid = accuracy_score(y_val, y_val_pred_grid)
print(f"Validation Accuracy (Grid Search): {val_acc_grid:.4f}")

In [ ]:
results_df = pd.DataFrame(grid_search.cv_results_)

print("All Grid Search Results:")
results_df[['param_model__n_neighbors', 'param_model__weights', 'param_model__metric', 'mean_test_score']]


### Randomized Search

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

param_distributions = {
    'model__n_neighbors': np.arange(1, 20),
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['euclidean', 'manhattan', 'chebyshev']
}

random_search = RandomizedSearchCV(
    estimator=combined_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

print("Best Parameters (Randomized Search):", random_search.best_params_)
print("Best Accuracy (Randomized Search):", random_search.best_score_)

best_pipeline_random = random_search.best_estimator_
y_val_pred_random = best_pipeline_random.predict(X_val)
val_acc_random = accuracy_score(y_val, y_val_pred_random)
print(f"Validation Accuracy (Randomized Search): {val_acc_random:.4f}")

In [ ]:
results_df = pd.DataFrame(random_search.cv_results_)

print("All Random Search Results:")
results_df[['param_model__n_neighbors', 'param_model__weights', 'param_model__metric', 'mean_test_score']]

In [ ]:
random_search.best_estimator_

In [ ]:
import pickle

best_pipeline_random = random_search.best_estimator_

with open('../best_pipeline_random.pkl', 'wb') as f:
    pickle.dump(best_pipeline_random, f)

print("Pipeline saved to 'best_pipeline_random.pkl'.")

### Evaluate your system on the Test Set

In [ ]:
from sklearn.metrics import accuracy_score

with open('../best_pipeline_random.pkl', 'rb') as f:
    loaded_pipeline = pickle.load(f)

print("Pipeline loaded successfully.")

y_test_pred = loaded_pipeline.predict(X_test)

test_acc = accuracy_score(y_test, y_test_pred)

print("\nEvaluation Results:")
print(f"Test Accuracy: {test_acc:.4f}")